In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.special import rel_entr
from scipy.stats import wasserstein_distance

## KL Divergence

In [2]:
def kl_divergence(p, q):
    """ Compute the Kullback-Leibler divergence between two probability distributions.

    Parameters
    ----------
    p : array-like
        The first probability distribution, the "true" distribution.
    q : array-like
        The second probability distribution, the "approximated" distribution.

    Returns
    -------
    float
        The Kullback-Leibler divergence between the two distributions.
    """
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return np.sum(rel_entr(p, q))

### Load extracted features from real dataset

In [2]:
real_feaures_file = "/mnt/local/work/filippo.schiazza2/Paper/Features/real_cell_features.csv"
df_real = pd.read_csv(real_feaures_file)

### Load extracted features from mask conditioned synthetic dataset

In [3]:
cond_gen_feaures_file = "/mnt/local/work/filippo.schiazza2/Paper/Features/cond_gen_cell_features.csv"
df_cond_gen = pd.read_csv(cond_gen_feaures_file)

### Load extracted features from unconditioned synthetic dataset

In [4]:
uncond_gen_feaures_file = "/mnt/local/work/filippo.schiazza2/Paper/Features/uncond_gen_cell_features.csv"
df_uncond_gen = pd.read_csv(uncond_gen_feaures_file)

### Load extracted features from 4Channels-unconditioned synthetic dataset

In [5]:
uncond4ch_gen_feaures_file = "/mnt/local/work/filippo.schiazza2/Paper/Features/4Channels_gen_cell_features.csv"
df_uncond4ch_gen = pd.read_csv(uncond4ch_gen_feaures_file)

## Feature histograms

In [ ]:
histograms_save_folder = '/mnt/local/work/filippo.schiazza2/Paper/Features/histograms/'
num_features = len(df_real.columns)

for i in tqdm(range(num_features)):
    print(df_real.columns[i])
    plt.figure()

    # Features
    real = df_real.iloc[:,i].values
    cond_gen = df_cond_gen.iloc[:,i].values
    uncond_gen = df_uncond_gen.iloc[:,i].values
    uncond4ch_gen = df_uncond4ch_gen.iloc[:,i].values

    # Histograms
    bins = np.histogram(np.hstack((real, cond_gen, uncond_gen, uncond4ch_gen)), bins=200)[1]
    plt.hist(real, bins=bins, alpha=0.5, label='Real', density=True)
    plt.hist(cond_gen, bins=bins, alpha=0.5, label='Conditional Gen', density=True)
    plt.hist(uncond_gen, bins=bins, alpha=0.5, label='Unconditional Gen', density=True)
    plt.hist(uncond4ch_gen, bins=bins, alpha=0.5, label='Unconditional Gen 4 Channels', density=True)
    plt.title(df_real.columns[i])
    plt.legend()
    plt.savefig(histograms_save_folder + df_real.columns[i] + '.png')
    plt.savefig(histograms_save_folder + df_real.columns[i] + '.svg')
    plt.close()

## Wasserstein Distance between features sets

In [10]:
columns = ['Feature Name', 
           'Wasserstein Real vs Conditional Gen',
           'Wasserstein Real vs Unconditional Gen',
           'Wasserstein Real vs 4Channels Gen',
           'Best Model',
           'Cond vs Uncond',
           'Cond vs Uncond 4 Channels']

wasserstein_df = pd.DataFrame(columns=columns)
num_features = len(df_real.columns)
for i in tqdm(range(num_features)):
    # remove the features with no meaning
    if df_real.columns[i] in ['Centroid X', 'Centroid Y', 'Min_red', 'Min_green', 'Min_blue', 'Max_red', 'Max_green', 'Max_blue', 'Min_hue', 'Min_saturation', 'Min_value', 'Max_hue', 'Max_saturation', 'Max_value']:
        continue

    # Features
    real = df_real.iloc[:,i].values
    cond_gen = df_cond_gen.iloc[:,i].values
    uncond_gen = df_uncond_gen.iloc[:,i].values
    uncond4ch_gen = df_uncond4ch_gen.iloc[:,i].values

    # Wasserstein distances
    wasserstein_real_cond_gen = wasserstein_distance(real, cond_gen)
    wasserstein_real_uncond_gen = wasserstein_distance(real, uncond_gen)
    wasserstein_real_uncond4ch_gen = wasserstein_distance(real, uncond4ch_gen)

    # Best model
    if wasserstein_real_cond_gen < wasserstein_real_uncond_gen and wasserstein_real_cond_gen < wasserstein_real_uncond4ch_gen:
        best_model = 1
    elif wasserstein_real_uncond_gen < wasserstein_real_cond_gen and wasserstein_real_uncond_gen < wasserstein_real_uncond4ch_gen:
        best_model = 2
    else:
        best_model = 3

    # Percentage improvement/deterioration
    cond_vs_uncond_image_only = (wasserstein_real_cond_gen - wasserstein_real_uncond_gen) / wasserstein_real_uncond_gen * 100
    cond_vs_uncond_image_mask = (wasserstein_real_cond_gen - wasserstein_real_uncond4ch_gen) / wasserstein_real_uncond4ch_gen * 100

    wasserstein_df.loc[i] = [df_real.columns[i],
                             wasserstein_real_cond_gen,
                             wasserstein_real_uncond_gen,
                             wasserstein_real_uncond4ch_gen,
                             best_model,
                             cond_vs_uncond_image_only,
                             cond_vs_uncond_image_mask]
    



100%|██████████| 49/49 [00:05<00:00,  9.64it/s]


In [11]:
wasserstein_df

,Feature Name,Wasserstein Real vs Conditional Gen,Wasserstein Real vs Unconditional Gen,Wasserstein Real vs 4Channels Gen,Best Model,Cond vs Uncond,Cond vs Uncond 4 Channels
2,Area,237.858753,198.500103,957.288521,2,19.828025,-75.152867
3,Perimeter,16.543199,14.436677,82.344554,2,14.591462,-79.909784
4,Equivalent Diameter,3.250104,5.039654,28.324619,1,-35.509380,-88.525515
5,Circularity,0.036098,0.087294,0.567911,1,-58.647901,-93.643761
6,Convexity,0.018114,0.052550,0.585041,1,-65.530536,-96.903862
7,Aspect Ratio,0.082950,0.067537,0.161447,2,22.821953,-48.621056
8,Minor Axis,2.837360,5.487256,27.948815,1,-48.291820,-89.848013
9,Major Axis,15283.860277,39955.140616,83912.784288,1,-61.747450,-81.786017
10,Eccentricity,0.023615,0.065290,0.201940,1,-63.831027,-88.306053
11,Extent,0.016936,0.072619,0.518724,1,-76.678679,-96.735108


In [9]:
# count the number of times each model is the best (express in percentage)
best_model_counts = wasserstein_df['Best Model'].value_counts()
best_model_counts = best_model_counts / best_model_counts.sum() * 100
best_model_counts

Best Model
1    85.714286
2    11.428571
3     2.857143
Name: count, dtype: float64

### Result comparison between Conditional and Unconditional (image only)

In [17]:
# count the number of times the conditional model is better than the unconditional model (express in percentage)
summary = wasserstein_df['Wasserstein Real vs Conditional Gen'] < wasserstein_df['Wasserstein Real vs Unconditional Gen']
summary = summary.value_counts()
summary = summary / summary.sum() * 100
summary


True     88.571429
False    11.428571
Name: count, dtype: float64

In [12]:
print(f'Total mean Wasserstein percentage change between cond vs uncond: {wasserstein_df['Cond vs Uncond'].mean():.2f}%')
# only when Wasserstein_real_cond_gen < Wasserstein_real_uncond_gen
print(f'Mean Wasserstein percentage improvement between cond vs uncond: {wasserstein_df[wasserstein_df['Cond vs Uncond'] < 0]['Cond vs Uncond'].mean():.2f}%')
# only when Wasserstein_real_cond_gen > Wasserstein_real_uncond_gen
print(f'Mean Wasserstein percentage deterioration between cond vs uncond: {wasserstein_df[wasserstein_df['Cond vs Uncond'] > 0]['Cond vs Uncond'].mean():.2f}%')

Total mean Wasserstein percentage change between cond vs uncond: -56.19%
Mean Wasserstein percentage improvement between cond vs uncond: -65.51%
Mean Wasserstein percentage deterioration between cond vs uncond: 16.03%


### Result comparison between Conditional and Unconditional (image + mask)

In [19]:
# count the number of times the conditional model is better than the unconditional model 4 channels (express in percentage)
summary = wasserstein_df['Wasserstein Real vs Conditional Gen'] < wasserstein_df['Wasserstein Real vs 4Channels Gen']
summary = summary.value_counts()
summary = summary / summary.sum() * 100
summary

True     97.142857
False     2.857143
Name: count, dtype: float64

In [13]:
print(f'Total mean Wasserstein percentage change between cond vs uncond 4 Channels: {wasserstein_df['Cond vs Uncond 4 Channels'].mean():.2f}%')
# only when Wasserstein_real_cond_gen < Wasserstein_real_uncond4ch_gen
print(f'Mean Wasserstein percentage improvement between cond vs uncond 4 Channels: {wasserstein_df[wasserstein_df['Cond vs Uncond 4 Channels'] < 0]['Cond vs Uncond 4 Channels'].mean():.2f}%')
# only when Wasserstein_real_cond_gen > Wasserstein_real_uncond4ch_gen
print(f'Mean Wasserstein percentage deterioration between cond vs uncond 4 Channels: {wasserstein_df[wasserstein_df['Cond vs Uncond 4 Channels'] > 0]['Cond vs Uncond 4 Channels'].mean():.2f}%')

Total mean Wasserstein percentage change between cond vs uncond 4 Channels: -80.94%
Mean Wasserstein percentage improvement between cond vs uncond 4 Channels: -83.34%
Mean Wasserstein percentage deterioration between cond vs uncond 4 Channels: 0.69%


## KL Divergence between Features sets

In [16]:
# Pandas dataframe for the KL divergences
columns = ['Feature Name', 
           'KL Real vs Conditional Gen', 
           'KL Real vs Unconditional Gen', 
           'KL_Outperform', 
           'KL_percentage_improvement',
           'Wasserstein Real vs Conditional Gen',
           'Wasserstein Real vs Unconditional Gen',
           'Wasserstein_Outperform',
           'Wasserstein_percentage_improvement']
metrics_df = pd.DataFrame(columns=columns)
num_bins = 200
num_features = len(df_real.columns)
for i in tqdm(range(num_features)):
    # Features
    real = df_real.iloc[:,i].values
    cond_gen = df_cond_gen.iloc[:,i].values
    uncond_gen = df_uncond_gen.iloc[:,i].values

    # Determine the common bin edges based on the combined range of all features
    min_edge = min(real.min(), cond_gen.min(), uncond_gen.min())
    max_edge = max(real.max(), cond_gen.max(), uncond_gen.max())
    bin_edges = np.linspace(min_edge, max_edge, num_bins + 1)

    # Compute histograms with the same bin edges
    hist_ref, _ = np.histogram(real, bins=bin_edges, density=True)
    hist_1, _ = np.histogram(cond_gen, bins=bin_edges, density=True)
    hist_2, _ = np.histogram(uncond_gen, bins=bin_edges, density=True)

    # Add a small value to avoid zero probabilities
    epsilon = 1e-10
    hist_ref += epsilon
    hist_1 += epsilon
    hist_2 += epsilon

    # Normalize the histograms to get probability distributions
    hist_ref /= np.sum(hist_ref)
    hist_1 /= np.sum(hist_1)
    hist_2 /= np.sum(hist_2)

    # Compute the Kullback-Leibler divergences
    kl_real_cond_gen = kl_divergence(hist_ref, hist_1)
    kl_real_uncond_gen = kl_divergence(hist_ref, hist_2)

    # Compute the percentage improvement
    kl_percentage_improvement = (kl_real_cond_gen - kl_real_uncond_gen) / kl_real_uncond_gen * 100

    # Wasserstein distance
    wasserstein_real_cond_gen = wasserstein_distance(real, cond_gen)
    wasserstein_real_uncond_gen = wasserstein_distance(real, uncond_gen)
    wass_percentage_improvement = (wasserstein_real_cond_gen - wasserstein_real_uncond_gen) / wasserstein_real_uncond_gen * 100

    metrics_df.loc[i] = [df_real.columns[i], 
                         kl_real_cond_gen, 
                         kl_real_uncond_gen, 
                         kl_real_cond_gen < kl_real_uncond_gen, 
                         kl_percentage_improvement,
                         wasserstein_real_cond_gen,
                         wasserstein_real_uncond_gen,
                         wasserstein_real_cond_gen < wasserstein_real_uncond_gen,
                         wass_percentage_improvement]

# remove the rows with names ['Centroid X', 'Centroid Y', 'Min_red', 'Min_green', 'Min_blue', 'Max_red', 'Max_green', 'Max_blue', 'Min_hue', 'Min_saturation', 'Min_value', 'Max_hue', 'Max_saturation', 'Max_value']
metrics_df = metrics_df[~metrics_df['Feature Name'].isin(['Centroid X', 'Centroid Y', 'Min_red', 'Min_green', 'Min_blue', 'Max_red', 'Max_green', 'Max_blue', 'Min_hue', 'Min_saturation', 'Min_value', 'Max_hue', 'Max_saturation', 'Max_value'])]

metrics_df.to_csv('/mnt/local/work/filippo.schiazza2/Paper/Features/kl_divergences.csv', index=False)
metrics_df


100%|██████████| 49/49 [00:03<00:00, 15.18it/s]


,Feature Name,KL Real vs Conditional Gen,KL Real vs Unconditional Gen,KL_Outperform,KL_percentage_improvement,Wasserstein Real vs Conditional Gen,Wasserstein Real vs Unconditional Gen,Wasserstein_Outperform,Wasserstein_percentage_improvement
2,Area,0.235940,0.078467,False,200.687325,237.858753,198.500103,False,19.828025
3,Perimeter,0.242717,0.089556,False,171.022980,16.543199,14.436677,False,14.591462
4,Equivalent Diameter,0.318002,0.095521,False,232.912314,3.250104,5.039654,True,-35.509380
5,Circularity,0.153647,0.112689,False,36.345993,0.036098,0.087294,True,-58.647901
6,Convexity,0.321546,0.195396,False,64.560900,0.018114,0.052550,True,-65.530536
7,Aspect Ratio,0.017769,0.017806,True,-0.208368,0.082950,0.067537,False,22.821953
8,Minor Axis,0.074787,0.097215,True,-23.070038,2.837360,5.487256,True,-48.291820
9,Major Axis,0.000034,0.000107,True,-68.653015,15283.860277,39955.140616,True,-61.747450
10,Eccentricity,0.035472,0.067927,True,-47.779364,0.023615,0.065290,True,-63.831027
11,Extent,0.048961,0.155344,True,-68.482178,0.016936,0.072619,True,-76.678679


### Fraction of features for which the conditioned model outperforms the unconditioned one

### KL analysis

In [8]:
metrics_df['KL_Outperform'].value_counts(normalize=True)

KL_Outperform
True     0.8
False    0.2
Name: proportion, dtype: float64

In [15]:
print(f'Total mean KL reduction: {metrics_df['KL_percentage_improvement'].mean():.2f}%')
# only when KL_real_cond_gen < KL_real_uncond_gen
print(f'Mean KL improvement: {metrics_df[metrics_df['KL_Outperform']]['KL_percentage_improvement'].mean():.2f}%')
# only when KL_real_cond_gen > KL_real_uncond_gen
print(f'Mean KL deterioration: {metrics_df[~metrics_df['KL_Outperform']]['KL_percentage_improvement'].mean():.2f}%')

Total mean KL reduction: -28.91%
Mean KL improvement: -65.39%
Mean KL deterioration: 116.99%


### Wasserstein analysis

In [17]:
metrics_df['Wasserstein_Outperform'].value_counts(normalize=True)

Wasserstein_Outperform
True     0.885714
False    0.114286
Name: proportion, dtype: float64

In [18]:
print(f'Total mean Wasserstein reduction: {metrics_df['Wasserstein_percentage_improvement'].mean():.2f}%')
# only when Wasserstein_real_cond_gen < Wasserstein_real_uncond_gen
print(f'Mean Wasserstein improvement: {metrics_df[metrics_df['Wasserstein_Outperform']]['Wasserstein_percentage_improvement'].mean():.2f}%')
# only when Wasserstein_real_cond_gen > Wasserstein_real_uncond_gen
print(f'Mean Wasserstein deterioration: {metrics_df[~metrics_df['Wasserstein_Outperform']]['Wasserstein_percentage_improvement'].mean():.2f}%')

Total mean Wasserstein reduction: -56.19%
Mean Wasserstein improvement: -65.51%
Mean Wasserstein deterioration: 16.03%
